In [1]:
import polars as pl

df = pl.scan_parquet("/Users/emiliodulay/Documents/1. UCLA/2. Year 2/3. Spring 2026/STAT M148/statM148proj/training_labeled_flattened.parquet")
print(df.head(1).collect())

shape: (1, 3)
┌──────────────────────┬─────────────────────────────────┬────────────┐
│ id                   ┆ journey                         ┆ is_success │
│ ---                  ┆ ---                             ┆ ---        │
│ str                  ┆ list[struct[2]]                 ┆ bool       │
╞══════════════════════╪═════════════════════════════════╪════════════╡
│ -635794259 449271353 ┆ [{2021-10-19 06:00:00 UTC,2}, … ┆ false      │
└──────────────────────┴─────────────────────────────────┴────────────┘


In [2]:
# Convert bool label → int  (True → 1, False → 0)
df = df.with_columns(
    pl.col("is_success").cast(pl.Int64).alias("label")
)

In [ ]:
# 1. Grab both columns and bring them into memory once
batch = df.select(["id", "label"]).collect()

# 2. Extract the Series and zip them
# In Polars, df["col_name"] returns a Series, which works perfectly in zip()
label_dict = dict(zip(batch["id"], batch["label"]))


# Split by id (never split mid-journey)
ids = batch["id"].to_list() # Creates a list of ids
split = int(len(ids) * 0.8) # 80% train, 20% val
train_ids, val_ids = set(ids[:split]), set(ids[split:]) # Creates a set of train ids and a set of validation ids

train_df = df.filter(pl.col("id").is_in(train_ids)).select(["id", "journey"]) # gets the ids and journeys corresponding to the train_ids
val_df   = df.filter(pl.col("id").is_in(val_ids)).select(["id", "journey"]) # get the ids and journeys corresponding to the val_ids

train_labels = {k: v for k, v in label_dict.items() if k in train_ids}
val_labels   = {k: v for k, v in label_dict.items() if k in val_ids}


from emilio_dataset import prepare_datasets
# Pass into the pipeline from dataset.py
train_loader, val_loader, _, scaler = prepare_datasets(
    train_df, val_df, val_df,   # using val as test placeholder for now
    train_labels, val_labels, val_labels,
    max_seq_len=256,
    batch_size=32,
)

# Build the model — binary classification = 2 classes
from emilio.transformers.time_series_transformer import make_classification_transformer

model = make_classification_transformer(
    num_features=7,    # however many features you extract per event
    num_classes=2,     # successful / not successful
)

## Full training Loop

In [ ]:
import torch
import torch.nn as nn

device    = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model     = model.to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-2)
criterion = nn.CrossEntropyLoss()

for epoch in range(5):
    model.train()
    for batch in train_loader:
        src   = batch["src"].to(device)           # (B, T, num_features)
        mask  = batch["padding_mask"].to(device)  # (B, T)
        label = batch["label"].to(device)         # (B,)  — 0 or 1

        logits = model(src, src_padding_mask=mask) # (B, 2)
        loss   = criterion(logits, label)

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

    # Validation
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for batch in val_loader:
            logits = model(
                batch["src"].to(device),
                src_padding_mask=batch["padding_mask"].to(device)
            )
            preds   = logits.argmax(dim=-1)       # (B,)
            correct += (preds == batch["label"].to(device)).sum().item()
            total   += len(preds)
    print(f"Epoch {epoch+1}  val accuracy: {correct/total:.3f}")

KeyboardInterrupt: 